# Reproduce the Prompt Injection Detector Benchmark

Run the full suite — **detection leaderboard**, **false positives on real traffic**, **indirect/structured injection**, and the **threshold-agnostic operating points** — on a **free Colab T4** (~15–30 min). Every number in the repo is produced by these scripts on public models and public datasets; this notebook lets you verify them, or add your own detector.

**Before you start:** set a GPU runtime — *Runtime → Change runtime type → T4 GPU*.

## 1. Get the code + install

In [ ]:
!git clone --quiet https://github.com/bastion-soft/pi-detector-bench.git
%cd pi-detector-bench
!pip install -q -e .

## 2. (Optional) Hugging Face login

The token is **optional**. A few baselines/datasets are **gated** and need a free HF token with access granted:
- `meta-llama/Prompt-Guard-86M` (a baseline) — accept on its model page
- `lmsys/lmsys-chat-1m` (an FPR dataset) — accept its license
- `hackaprompt/hackaprompt-dataset` (an indirect set) — accept on its dataset page

Without a token these are **excluded — skipped cleanly** and the rest of the run continues unaffected. The free 70M model and every open baseline/dataset need no token.

In [ ]:
# Optional: add HF_TOKEN to Colab Secrets (key icon, left), or skip this cell.
from huggingface_hub import login

try:
    from google.colab import userdata

    login(userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret HF_TOKEN.")
except Exception:
    print("No HF_TOKEN found - gated models/datasets will be skipped (that's fine).")

## 3. Detection leaderboard

AUC + F1 across four held-out adversarial benchmarks (rogue, xTRam1, S-Labs, JailbreakBench). The script prints a markdown table and writes `results/leaderboard.{json,md}`.

Tip: add `--limit 200` for a fast smoke run first.

In [ ]:
!python -m scripts.run_leaderboard --dump-scores results/scores

## 4. False-positive rate

The half most comparisons skip: the share of **benign** real-user messages each model wrongly flags, on WildChat + LMSYS openers. Lower is better. Writes `results/false_positives.json`.

Tip: add `--n 500` for a fast smoke run first.

In [ ]:
!python -m scripts.measure_false_positives --dump-scores results/scores

## 5. Indirect / structured injection

Where most detectors fall off: injection hidden inside **data** — JSON/XML agent interactions (Z-Edgar), document context (BIPIA), poisoned tool outputs (InjecAgent), and agentic attacks (AgentDojo / HackAPrompt / TensorTrust). Scored pure-model, same as §3, on a **separate** table (a distinct capability axis — not folded into the §3 average). Writes `results/indirect.{json,md}`.

HackAPrompt is gated (needs the HF token from §2); AgentDojo needs `pip install agentdojo` — both skip cleanly if unavailable.

In [ ]:
# AgentDojo ships as its own package (one of the indirect sets); install it here.
!pip install -q agentdojo
!python -m scripts.eval_indirect --dump-scores results/scores_indirect

## 6. Operating points (threshold-agnostic) + reproducible tables

First, `rebuild_results_from_scores` recomputes the leaderboard + indirect tables **from the dumped scores**, so every published number is scores-derived and doesn't depend on GPU run-to-run drift (the torch baselines vary slightly between runs; Bastion's ONNX path is deterministic).

Then the fairest comparison: instead of a fixed 0.5 line, set each detector's threshold to the **same detection rate** (e.g. catch 95% of attacks) and report the cost in false alarms — so the ranking doesn't depend on where anyone's 0.5 falls. Two views:

- **Direct** — threshold from the attack sets, FPR on **real benign traffic** (WildChat + LMSYS); plus EER and the 0.2/0.45/0.5/0.55/0.8 sweep.
- **Indirect / structured** — within each matched set, FPR on **benign structured records** at a fixed catch rate ("does it catch the injection without flagging legitimate JSON / invoices / tool output?").

All of §6 is pure post-processing over the scores dumped in §3–§5 (no GPU, fully reproducible). Writes `operating_points{,_indirect}.{json,md}`, `det_points{,_indirect}.json`, and the curve SVG/PNGs. See the verdict in [`results/FINDINGS.md`](results/FINDINGS.md).

In [ ]:
# First, make every published table reproducible from the dumped scores — recompute
# AUC/F1/etc from scores/ so the tables don't depend on GPU run-to-run drift (no GPU).
!python -m scripts.rebuild_results_from_scores

# Threshold-agnostic analysis over the per-prompt scores dumped in §3 + §4 (no GPU).
# Direct: threshold from attacks, FPR on real benign traffic.
!python -m scripts.analyze_operating_points
# Indirect/structured: within each set, FPR on benign structured records at a fixed catch rate.
!python -m scripts.analyze_operating_points --scores-dir results/scores_indirect --within-set --label indirect
# Optional curves (DET/operating + FPR-vs-threshold):
!pip install -q matplotlib && python -m scripts.plot_operating_points

## Results

- `results/leaderboard.json` + `leaderboard.md` — detection (AUC / F1 / latency)
- `results/false_positives.json` — false-positive rate at the fixed 0.5 threshold
- `results/indirect.{json,md}` — indirect / structured injection (AUC / F1)
- `results/operating_points.{json,md}` — direct threshold-agnostic view: FPR at a fixed detection rate, EER, 0.2/0.45/0.5/0.55/0.8 sweep
- `results/operating_points_indirect.{json,md}` — indirect within-set: FPR on benign structured records at a fixed catch rate
- `results/det_points{,_indirect}.json` (+ optional `*.svg`/`*.png`) — operating / threshold curves
- `results/scores/` + `scores_indirect/` — raw per-prompt scores, so every number above is reproducible offline

Docs: [methodology](https://github.com/bastion-soft/pi-detector-bench/blob/main/METHODOLOGY.md) · [honest verdict on the results](https://github.com/bastion-soft/pi-detector-bench/blob/main/results/FINDINGS.md) · [add your detector](https://github.com/bastion-soft/pi-detector-bench/blob/main/CONTRIBUTING.md).